In [3]:
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string

# ---- SETTINGS ----
input_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/INV clean file to learn.xlsx'
output_mapping_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic-INV.xlsx'
id_columns = ['Variable', 'Sector', 'Unit', 'Unnamed: 3']
year_col_map = {5 + i: year for i, year in enumerate(range(2025, 2055, 5))}  # E to J → 2025 to 2050

# ---- Load Excel workbook ----
wb = load_workbook(input_file, data_only=False)
ws_ind = wb["IND"]
ind_df = pd.read_excel(input_file, sheet_name="IND")
agg_df = pd.read_excel(input_file, sheet_name="Aggregated_By_Sector")
multi_df = pd.read_excel(input_file, sheet_name="Final_Investments_Multiplied")

# ---- Extract cell references (handles ranges and subtract) ----
def extract_formula_refs(formula):
    cleaned = formula.strip().lstrip('=').replace("'", "")
    pattern = r'([A-Za-z0-9 _&,\-]+)!([A-Z]+\d+)(?::([A-Z]+\d+))?'  # Match single or range
    refs = []
    for match in re.findall(pattern, cleaned):
        sheet, start, end = match
        if end:
            # range: expand to all rows between start and end
            col_letter = re.match(r"[A-Z]+", start).group()
            start_row = int(re.search(r"\d+", start).group())
            end_row = int(re.search(r"\d+", end).group())
            for r in range(start_row, end_row + 1):
                refs.append((sheet.strip(), f"{col_letter}{r}"))
        else:
            refs.append((sheet.strip(), start))
    return refs

# ---- Build Mapping ----
mapping = []

for row_idx in range(2, ws_ind.max_row + 1):
    try:
        ind_row = ind_df.iloc[row_idx - 2]
        row_id = {col: ind_row[col] for col in id_columns}
    except:
        continue

    for col_idx in range(5, 11):  # E to J
        formula = ws_ind.cell(row=row_idx, column=col_idx).value
        if not isinstance(formula, str) or not formula.startswith("="):
            continue

        year = year_col_map[col_idx]
        multiplier = -1000 if "*-1000" in formula else 1000 if "*1000" in formula else 1

        refs = extract_formula_refs(formula)
        for sheet_name, cell_ref in refs:
            try:
                col_letter, row_num = coordinate_from_string(cell_ref)
                col_index = column_index_from_string(col_letter) - 1  # 0-based
                row_num = int(row_num)

                if sheet_name == "Aggregated_By_Sector":
                    row_data = agg_df.iloc[row_num - 2]
                    carrier = row_data["Energy Carrier"]
                    sector = row_data["Sector"]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "aggregated_sector",
                        "carrier": carrier,
                        "sector": sector,
                        "technology": "",
                        "source_block": "",
                        "multiplier": multiplier
                    })

                elif sheet_name == "Final_Investments_Multiplied":
                    row_data = multi_df.iloc[row_num - 2]
                    tech = row_data["Technology"]
                    carrier = row_data["Energy Carrier"]
                    sector = row_data["Sector"]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "final_investment",
                        "technology": tech,
                        "carrier": carrier,
                        "sector": sector,
                        "source_block": "",
                        "multiplier": multiplier
                    })

            except:
                continue

# ---- Save Mapping ----
mapping_df = pd.DataFrame(mapping).drop_duplicates()
mapping_df.to_excel(output_mapping_file, index=False)
print(f"✔ Mapping saved to: {output_mapping_file}")


✔ Mapping saved to: /Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic-INV.xlsx


In [9]:
import pandas as pd
from openpyxl import load_workbook

# ---- SETTINGS ----
mapping_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic-INV.xlsx'
source_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx'
sheet_name_to_add = 'IND_rebuilt'

# ---- Load mapping and source data ----
mapping_df = pd.read_excel(mapping_file)
agg_df = pd.read_excel(source_file, sheet_name="Aggregated_By_Sector")
final_inv_df = pd.read_excel(source_file, sheet_name="Final_Investments_Multiplied")

# ---- Setup ----
id_cols = ['Variable', 'Sector', 'Unit', 'Unnamed: 3']
all_years = sorted(mapping_df['year'].unique())
reconstructed = []

# ---- Recalculate based on mapping ----
for key, group in mapping_df.groupby(id_cols):
    row = dict(zip(id_cols, key))

    for year in all_years:
        year_val = 0
        year_group = group[group["year"] == year]
        year_column = f"{year}_investment_multiplied"

        for _, entry in year_group.iterrows():
            val = 0

            try:
                if entry["source_type"] == "aggregated_sector":
                    source_row = agg_df[
                        (agg_df["Energy Carrier"] == entry["carrier"]) &
                        (agg_df["Sector"] == entry["sector"])
                    ]
                    if not source_row.empty:
                        val = source_row.iloc[0].get(year_column, 0)

                elif entry["source_type"] == "final_investment":
                    source_row = final_inv_df[
                        (final_inv_df["Technology"] == entry["technology"]) &
                        (final_inv_df["Energy Carrier"] == entry["carrier"]) &
                        (final_inv_df["Sector"] == entry["sector"])
                    ]
                    if not source_row.empty:
                        val = source_row.iloc[0].get(year_column, 0)

                if pd.isna(val):
                    val = 0

                year_val += val * entry["multiplier"]

            except Exception as e:
                continue  # Silent fail per entry

        row[str(year)] = year_val

    reconstructed.append(row)

# ---- Save result to Excel ----
reconstructed_df = pd.DataFrame(reconstructed)
reconstructed_df = reconstructed_df[id_cols + [str(y) for y in all_years]]

with pd.ExcelWriter(source_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    reconstructed_df.to_excel(writer, sheet_name=sheet_name_to_add, index=False)

print(f"✔ Rebuilt IND sheet saved in '{sheet_name_to_add}' of: {source_file}")


✔ Rebuilt IND sheet saved in 'IND_rebuilt' of: /Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx


In [ ]:
# ignore the below code

In [ ]:
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.cell import coordinate_from_string

# ---- SETTINGS ----
input_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/INV clean file to learn.xlsx'
output_mapping_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic-INV2.xlsx'
id_columns = ['Variable', 'Sector', 'Unit', 'Unnamed: 3']
year_col_map = {5 + i: year for i, year in enumerate(range(2025, 2055, 5))}  # E to J

# ---- Load Excel workbook ----
wb = load_workbook(input_file, data_only=False)
ws_ind = wb["IND"]
ind_df = pd.read_excel(input_file, sheet_name="IND")
agg_df = pd.read_excel(input_file, sheet_name="Aggregated_By_Sector")
finv_df = pd.read_excel(input_file, sheet_name="Final_Investments_Multiplied")

# ---- Extract formula references ----
def extract_all_formula_references(formula_string):
    formula_string = formula_string.strip().lstrip("=").replace("'", "")
    refs = []

    # Handle SUM() ranges
    sum_matches = re.findall(r'SUM\(([A-Za-z0-9 _\-]+)!([A-Z]{1,3})([0-9]+):([A-Z]{1,3})([0-9]+)\)', formula_string)
    for sheet, col_start, row_start, col_end, row_end in sum_matches:
        for r in range(int(row_start), int(row_end) + 1):
            refs.append((sheet.strip(), f"{col_start}{r}"))

    # Handle individual cell references (outside SUM)
    single_refs = re.findall(r'([A-Za-z0-9 _\-]+)!([A-Z]{1,3}[0-9]{1,5})', formula_string)
    for sheet, cell in single_refs:
        if (sheet.strip(), cell) not in refs:
            refs.append((sheet.strip(), cell))

    return refs

# ---- Build mapping ----
mapping = []

for row_idx in range(2, ws_ind.max_row + 1):
    try:
        ind_row = ind_df.iloc[row_idx - 2]
        row_id = {col: ind_row[col] for col in id_columns}
    except:
        continue

    for col_idx in range(5, 11):  # E to J = years 2025–2050
        formula = ws_ind.cell(row=row_idx, column=col_idx).value
        if not isinstance(formula, str) or not formula.startswith("="):
            continue

        year = year_col_map.get(col_idx)
        multiplier = -1000 if "*-1000" in formula else 1000 if "*1000" in formula else 1

        refs = extract_all_formula_references(formula)
        for sheet_name, cell_ref in refs:
            try:
                col_letter, ref_row = coordinate_from_string(cell_ref)
                ref_row = int(ref_row)
            except:
                continue

            sheet_name = sheet_name.strip()
            try:
                if sheet_name == "Aggregated_By_Sector":
                    row_data = agg_df.iloc[ref_row - 2]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "aggregated_sector",
                        "carrier": row_data["Energy Carrier"],
                        "sector": row_data["Sector"],
                        "technology": "",
                        "source_block": "",
                        "multiplier": multiplier
                    })

                elif sheet_name == "Final_Investments_Multiplied":
                    row_data = finv_df.iloc[ref_row - 2]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "final_investment",
                        "carrier": row_data["Energy Carrier"],
                        "sector": row_data["Sector"],
                        "technology": row_data["Technology"],
                        "source_block": "",
                        "multiplier": multiplier
                    })

            except:
                continue

# ---- Save Mapping ----
mapping_df = pd.DataFrame(mapping).drop_duplicates()
mapping_df.to_excel(output_mapping_file, index=False)
print(f"✔ Mapping saved to: {output_mapping_file}")
